In [0]:
from pyspark.sql import functions as F

CATALOG = "rio_dataengineering"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")
spark.sql(f"USE SCHEMA {SILVER_SCHEMA}")

print("Silver transformation configuration ready.")

In [0]:
def write_silver_table(df, table_name: str) -> int:
    """
    Overwrite a Silver Delta table and return its row count.
    """
    target_table = f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    return df.count()

In [0]:
departments_silver = (
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.departments")
    .select(
        F.upper(F.trim("department_id")).alias("department_id"),
        F.when(
            F.upper(F.trim("department_name")).isin("HR", "HSE"),
            F.upper(F.trim("department_name"))
        )
        .otherwise(F.initcap(F.trim("department_name")))
        .alias("department_name")
    )
    .filter(F.col("department_id").isNotNull())
    .dropDuplicates(["department_id"])
)

department_rows = write_silver_table(
    departments_silver,
    "departments"
)

print(f"Loaded silver.departments: {department_rows} rows")

In [0]:
locations_silver = (
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.locations")
    .select(
        F.upper(F.trim("location_id")).alias("location_id"),
        F.initcap(F.trim("city")).alias("city"),
        F.initcap(F.trim("country")).alias("country"),
        F.upper(F.trim("region")).alias("region")
    )
    .filter(F.col("location_id").isNotNull())
    .dropDuplicates(["location_id"])
)

location_rows = write_silver_table(
    locations_silver,
    "locations"
)

print(f"Loaded silver.locations: {location_rows} rows")

In [0]:
employees_silver = (
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.employees")
    .select(
        F.upper(F.trim("employee_id")).alias("employee_id"),
        F.initcap(F.trim("first_name")).alias("first_name"),
        F.initcap(F.trim("last_name")).alias("last_name"),
        F.initcap(F.trim("gender")).alias("gender"),
        F.to_date("birth_date").alias("birth_date"),
        F.to_date("hire_date").alias("hire_date"),
        F.to_date("termination_date").alias("termination_date"),
        F.initcap(F.trim("employment_status")).alias("employment_status"),
        F.initcap(F.trim("employment_type")).alias("employment_type"),
        F.upper(F.trim("department_id")).alias("department_id"),
        F.upper(F.trim("location_id")).alias("location_id"),
        F.initcap(F.trim("business_unit")).alias("business_unit"),
        F.upper(F.trim("salary_band")).alias("salary_band"),
        F.lower(F.trim("email")).alias("email"),
        F.upper(F.trim("manager_id")).alias("manager_id")
    )
    .filter(F.col("employee_id").isNotNull())
    .dropDuplicates(["employee_id"])
    .withColumn(
        "full_name",
        F.concat_ws(" ", F.col("first_name"), F.col("last_name"))
    )
    .withColumn(
        "age_years",
        F.floor(
            F.months_between(
                F.current_date(),
                F.col("birth_date")
            ) / 12
        )
    )
    .withColumn(
        "tenure_years",
        F.round(
            F.months_between(
                F.coalesce(
                    F.col("termination_date"),
                    F.current_date()
                ),
                F.col("hire_date")
            ) / 12,
            2
        )
    )
    .withColumn(
        "is_active",
        (
            (F.col("employment_status") == "Active")
            & F.col("termination_date").isNull()
        )
    )
    .withColumn(
        "status_quality_flag",
        F.when(
            (F.col("employment_status") == "Active")
            & F.col("termination_date").isNotNull(),
            "ACTIVE_WITH_TERMINATION_DATE"
        )
        .when(
            (F.col("employment_status") == "Terminated")
            & F.col("termination_date").isNull(),
            "TERMINATED_WITHOUT_DATE"
        )
        .otherwise("VALID")
    )
)

employee_rows = write_silver_table(
    employees_silver,
    "employees"
)

print(f"Loaded silver.employees: {employee_rows} rows")

In [0]:
payroll_silver = (
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.payroll")
    .select(
        F.upper(F.trim("employee_id")).alias("employee_id"),
        F.to_date("payroll_month").alias("payroll_month"),
        F.col("base_salary")
         .cast("decimal(12,2)")
         .alias("base_salary"),
        F.col("overtime")
         .cast("decimal(12,2)")
         .alias("overtime"),
        F.col("bonus")
         .cast("decimal(12,2)")
         .alias("bonus"),
        F.col("tax")
         .cast("decimal(12,2)")
         .alias("tax"),
        F.col("net_pay")
         .cast("decimal(12,2)")
         .alias("net_pay")
    )
    .filter(
        F.col("employee_id").isNotNull()
        & F.col("payroll_month").isNotNull()
    )
    .dropDuplicates(["employee_id", "payroll_month"])
    .withColumn(
        "gross_pay",
        F.col("base_salary")
        + F.col("overtime")
        + F.col("bonus")
    )
)

payroll_rows = write_silver_table(
    payroll_silver,
    "payroll"
)

print(f"Loaded silver.payroll: {payroll_rows} rows")

In [0]:
training_silver = (
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.training")
    .select(
        F.upper(F.trim("employee_id")).alias("employee_id"),
        F.initcap(F.trim("course_name")).alias("course_name"),
        F.to_date("completion_date").alias("completion_date"),
        F.col("hours")
         .cast("integer")
         .alias("training_hours"),
        F.col("score")
         .cast("integer")
         .alias("score"),
        F.col("mandatory")
         .cast("boolean")
         .alias("is_mandatory")
    )
    .filter(
        F.col("employee_id").isNotNull()
        & F.col("course_name").isNotNull()
    )
    .dropDuplicates(
        ["employee_id", "course_name", "completion_date"]
    )
    .withColumn(
        "passed",
        F.col("score") >= 60
    )
    .withColumn(
        "training_quality_flag",
        F.when(
            ~F.col("score").between(0, 100),
            "INVALID_SCORE"
        )
        .when(
            F.col("training_hours") <= 0,
            "INVALID_HOURS"
        )
        .otherwise("VALID")
    )
)

training_rows = write_silver_table(
    training_silver,
    "training"
)

print(f"Loaded silver.training: {training_rows} rows")

In [0]:
engagement_silver = (
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.engagement")
    .select(
        F.upper(F.trim("employee_id")).alias("employee_id"),
        F.to_date("survey_date").alias("survey_date"),
        F.col("engagement_score")
         .cast("integer")
         .alias("engagement_score"),
        F.col("wellbeing_score")
         .cast("integer")
         .alias("wellbeing_score"),
        F.col("manager_score")
         .cast("integer")
         .alias("manager_score")
    )
    .filter(
        F.col("employee_id").isNotNull()
        & F.col("survey_date").isNotNull()
    )
    .dropDuplicates(["employee_id", "survey_date"])
    .withColumn(
        "overall_score",
        F.round(
            (
                F.col("engagement_score")
                + F.col("wellbeing_score")
                + F.col("manager_score")
            ) / 3,
            2
        )
    )
    .withColumn(
        "engagement_category",
        F.when(F.col("overall_score") >= 8, "High")
        .when(F.col("overall_score") >= 5, "Medium")
        .otherwise("Low")
    )
    .withColumn(
        "engagement_quality_flag",
        F.when(
            ~F.col("engagement_score").between(1, 10)
            | ~F.col("wellbeing_score").between(1, 10)
            | ~F.col("manager_score").between(1, 10),
            "INVALID_SCORE"
        )
        .otherwise("VALID")
    )
)

engagement_rows = write_silver_table(
    engagement_silver,
    "engagement"
)

print(f"Loaded silver.engagement: {engagement_rows} rows")

In [0]:
leave_silver = (
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.leave")
    .select(
        F.upper(F.trim("employee_id")).alias("employee_id"),
        F.initcap(F.trim("leave_type")).alias("leave_type"),
        F.to_date("start_date").alias("start_date"),
        F.col("days")
         .cast("integer")
         .alias("leave_days"),
        F.col("approved")
         .cast("boolean")
         .alias("approved")
    )
    .filter(
        F.col("employee_id").isNotNull()
        & F.col("start_date").isNotNull()
    )
    .dropDuplicates(
        ["employee_id", "leave_type", "start_date"]
    )
    .withColumn(
        "end_date",
        F.date_add(
            F.col("start_date"),
            F.col("leave_days") - 1
        )
    )
    .withColumn(
        "leave_quality_flag",
        F.when(
            F.col("leave_days") <= 0,
            "INVALID_LEAVE_DAYS"
        )
        .otherwise("VALID")
    )
)

leave_rows = write_silver_table(
    leave_silver,
    "leave"
)

print(f"Loaded silver.leave: {leave_rows} rows")

In [0]:
employee_profile_silver = (
    spark.table(f"{CATALOG}.{SILVER_SCHEMA}.employees").alias("e")
    .join(
        spark.table(
            f"{CATALOG}.{SILVER_SCHEMA}.departments"
        ).alias("d"),
        F.col("e.department_id") == F.col("d.department_id"),
        "left"
    )
    .join(
        spark.table(
            f"{CATALOG}.{SILVER_SCHEMA}.locations"
        ).alias("l"),
        F.col("e.location_id") == F.col("l.location_id"),
        "left"
    )
    .select(
        "e.*",
        F.col("d.department_name"),
        F.col("l.city"),
        F.col("l.country"),
        F.col("l.region")
    )
)

profile_rows = write_silver_table(
    employee_profile_silver,
    "employee_profile"
)

print(f"Loaded silver.employee_profile: {profile_rows} rows")

In [0]:
SILVER_TABLES = [
    "employees",
    "employee_profile",
    "departments",
    "locations",
    "payroll",
    "training",
    "engagement",
    "leave"
]

validation_results = []

for table_name in SILVER_TABLES:
    df = spark.table(
        f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
    )

    validation_results.append(
        {
            "table_name": table_name,
            "row_count": df.count(),
            "column_count": len(df.columns),
            "status": "PASS" if df.count() > 0 else "FAIL"
        }
    )

validation_df = spark.createDataFrame(validation_results)

display(validation_df.orderBy("table_name"))

In [0]:
quality_checks = [
    (
        "employee_status",
        spark.table(
            f"{CATALOG}.{SILVER_SCHEMA}.employees"
        )
        .filter(F.col("status_quality_flag") != "VALID")
        .count()
    ),
    (
        "training",
        spark.table(
            f"{CATALOG}.{SILVER_SCHEMA}.training"
        )
        .filter(F.col("training_quality_flag") != "VALID")
        .count()
    ),
    (
        "engagement",
        spark.table(
            f"{CATALOG}.{SILVER_SCHEMA}.engagement"
        )
        .filter(F.col("engagement_quality_flag") != "VALID")
        .count()
    ),
    (
        "leave",
        spark.table(
            f"{CATALOG}.{SILVER_SCHEMA}.leave"
        )
        .filter(F.col("leave_quality_flag") != "VALID")
        .count()
    ),
    (
        "missing_department_join",
        spark.table(
            f"{CATALOG}.{SILVER_SCHEMA}.employee_profile"
        )
        .filter(F.col("department_name").isNull())
        .count()
    ),
    (
        "missing_location_join",
        spark.table(
            f"{CATALOG}.{SILVER_SCHEMA}.employee_profile"
        )
        .filter(F.col("city").isNull())
        .count()
    )
]

quality_df = spark.createDataFrame(
    [
        (
            check_name,
            failure_count,
            "PASS" if failure_count == 0 else "FAIL"
        )
        for check_name, failure_count in quality_checks
    ],
    ["quality_check", "failure_count", "status"]
)

display(quality_df.orderBy("quality_check"))

failed_checks = [
    check_name
    for check_name, failure_count in quality_checks
    if failure_count > 0
]

if failed_checks:
    raise ValueError(
        f"Silver quality checks failed: {failed_checks}"
    )

print("Silver transformations and quality checks completed successfully.")